# Tarea 4: intervalos de confianza, construcción de Neyman y Feldman–Cousins

**Objetivo.** Explorar la construcción de intervalos de confianza frecuentistas: la construcción de Neyman y la técnica de Feldman–Cousins (FC) cuando el parámetro tiene una frontera física

**Convenciones.**

- Todas las cantidades gaussianas están en unidades de $\sigma$ ($\sigma = 1$).
- Nivel de confianza $\mathrm{CL} = 0.90$, definido en la celda de configuración; los enunciados suponen este valor.
- $\Phi$ denota la función de distribución acumulada de la normal estándar.



**Contenido.**

1. Cobertura de un intervalo sin frontera.
2. Construcción de Neyman sin frontera.
3. Frontera física $\mu \ge 0$ con el cinturón central.
4. Feldman–Cousins para $\mu \ge 0$.

In [ ]:
import numpy as np                     
import matplotlib.pyplot as plt       
from scipy.stats import norm           

rng = np.random.default_rng(seed=2026)

SIGMA = 1.0      # Gaussian resolution (everything is in units of sigma)
CL = 0.90        # confidence level (90 %)

print(f"SIGMA = {SIGMA}, CL = {CL}")

## Ejercicio 1: el intervalo central y su cobertura

$\mu$ puede ser cualquier número real. Una medición da $x \sim \mathcal{N}(\mu, \sigma^2)$ y reportamos el intervalo central

$$
[x - z\sigma,\; x + z\sigma].
$$

a) Calcula $z$ para $\mathrm{CL} = 90\,\%$ y guárdalo en una variable `z`; los siguientes ejercicios la usan.

b) Simula $N = 10\,000$ mediciones con $\mu_{\text{true}} = 1$. ¿En qué fracción de ellas el intervalo contiene a $\mu_{\text{true}}$? ¿Es compatible con 0.90 dentro de su error estándar?

c) Repite b) con $\mu_{\text{true}} = -2$ y $\mu_{\text{true}} = 3$. ¿Depende la cobertura de $\mu_{\text{true}}$? En la frase «$\mu$ está en $[x - z\sigma,\; x + z\sigma]$ con 90 % de probabilidad», ¿qué es aleatorio: $\mu$ o el intervalo?



Valor de $z$:

$$
P\big(|x - \mu| \le z\sigma\big) = \mathrm{CL} \quad\Longrightarrow\quad z = \Phi^{-1}\!\left(\frac{1}{2} + \frac{\mathrm{CL}}{2}\right)
$$

Cobertura estimada y su error estándar:

$$
\widehat{\mathrm{cov}} = \frac{\text{número de intervalos que contienen a } \mu_{\text{true}}}{N}, \qquad \mathrm{SE} = \sqrt{\frac{\mathrm{CL}\,(1 - \mathrm{CL})}{N}}
$$

### hintss

- `norm.ppf(p)` calcula $\Phi^{-1}(p)$.
- `rng.normal(loc=mu_true, scale=SIGMA, size=N)` genera las $N$ mediciones.
- Comparar arreglos da un arreglo de `True`/`False`, y `np.mean(...)` sobre él da la fracción de `True`. Para combinar dos condiciones usa `&` con paréntesis: `(a <= b) & (b <= c)`.



In [ ]:
# Exercise 1 - Solution here

## Ejercicio 2: la construcción de Neyman


1. **Antes de medir.** Para cada valor posible de $\mu$ se elige una *región de aceptación* $A(\mu)$ que contiene a $x$ con probabilidad $\mathrm{CL}$. Aquí usamos la región central, $A(\mu) = [\mu - z\sigma,\; \mu + z\sigma]$.
2. **El cinturón.** Al dibujar todas las $A(\mu)$ en el plano ($x$ horizontal, $\mu$ vertical) se obtiene el *cinturón de confianza*.
3. **Después de medir $x_0$.** El intervalo de confianza contiene todos los $\mu$ cuya región de aceptación contiene a $x_0$. En la gráfica, es el tramo de la recta vertical $x = x_0$ que queda dentro del cinturón:

$$
C(x_0) = \{\mu : x_0 \in A(\mu)\}
$$

a) Dibuja el cinturón para $\mu \in [-3, 5]$.

b) Para $x_0 = 1.5$, selecciona numéricamente los valores de la malla de $\mu$ que pertenecen a $C(x_0)$ y reporta el mínimo y el máximo. Dibuja la recta $x = x_0$ sobre el cinturón.

c) Demuestra analíticamente que $C(x_0) = [x_0 - z\sigma,\; x_0 + z\sigma]$ y compara con b).


$$
x_0 \in A(\mu) \iff \mu - z\sigma \le x_0 \le \mu + z\sigma \iff |x_0 - \mu| \le z\sigma
$$


In [ ]:
# Exercise 2 - Solution here

## Ejercicio 3: la frontera física $\mu \ge 0$

Ahora $\mu$ es restringida y no puede ser negativa. La medición $x_0$ sí puede salir negativa, por una fluctuación estadísticas. Repetimos la construcción del ejercicio 2 solo para $\mu \ge 0$:

$$
C(x_0) = \{\mu \ge 0 : |x_0 - \mu| \le z\sigma\}
$$

a) Dibuja el cinturón para $\mu \in [0, 5]$ y calcula $C(x_0)$ para $x_0 \in \{-2,\; -1,\; 0,\; 1\}$. ¿Alguno es vacío?

b) Demuestra que $C(x_0)$ es vacío si y solo si $x_0 < -z\sigma$. Si $\mu_{\text{true}} = 0$, ¿con qué probabilidad sale un intervalo vacío? Calcúlala con la fórmula de abajo y verifícala simulando $N = 10\,000$ mediciones.

c) ¿Un intervalo vacío significa que la medición está mal? ¿Qué reportarías en ese caso? ¿Por qué tampoco es aceptable reportar un intervalo con valores negativos de $\mu$?

In [ ]:
# Exercise 3 - Solution here

## Ejercicio 4: Feldman–Cousins

El problema del ejercicio 3 viene de cómo elegimos $A(\mu)$: siempre la región central, sin tomar en cuenta la frontera. Feldman y Cousins proponen otra regla para decidir qué valores de $x$ entran primero en $A(\mu)$. Esa regla usa el cociente de verosimilitudes

$$
R(x;\mu) = \frac{f(x \mid \mu)}{f\big(x \mid \hat\mu(x)\big)}, \qquad \hat\mu(x) = \max(0,\, x),
$$

donde $\hat\mu(x)$ es el valor físicamente permitido de $\mu$ que mejor explica $x$. Para la gaussiana:

$$
R(x;\mu) = \exp\left\{-\frac{(x - \mu)^2 - \big(x - \hat\mu(x)\big)^2}{2\sigma^2}\right\}, \qquad 0 < R \le 1
$$

**Receta.** Para cada $\mu$ de la malla:

1. Calcula la probabilidad de cada punto de la malla de $x$: $p_i = f(x_i \mid \mu)\, \Delta x$.
2. Calcula $R(x_i;\mu)$.
3. Ordena los puntos de mayor a menor $R$.
4. Suma sus probabilidades en ese orden hasta alcanzar $\mathrm{CL}$. Los puntos sumados forman $A(\mu)$; guarda su mínimo y su máximo.

Después se invierte igual que antes: $C(x_0) = \{\mu \ge 0 : x_0 \in A(\mu)\}$.

a) Implementa la receta y dibuja el cinturón FC junto al cinturón central del ejercicio 3.

b) Calcula el intervalo FC para $x_0 \in \{-2,\; -1,\; 0,\; 1,\; 2\}$ y haz una tabla que lo compare con el intervalo del ejercicio 3. ¿Alguno es vacío?

c) Explica por qué el intervalo FC nunca es vacío. *Pista:* ¿cuánto vale $R\big(x_0;\, \hat\mu(x_0)\big)$? Entonces, ¿entra $x_0$ en $A\big(\hat\mu(x_0)\big)$?

d) Para $x_0$ pequeño, FC da un límite superior, $[0,\; \mu_{\text{up}}]$. Para $x_0$ grande, da un intervalo de dos lados, $[\mu_{\text{lo}},\; \mu_{\text{up}}]$ con $\mu_{\text{lo}} > 0$. ¿Aproximadamente a partir de qué $x_0$ ocurre el cambio? ¿Por qué conviene que el método lo decida, en lugar de que el físico elija después de ver el dato (el problema conocido como *flip-flopping*)?

### hints

- Mallas: `dx = 0.01`, `x_grid = np.round(np.arange(-6, 10.001, dx), 2)`, `mu_grid = np.round(np.arange(0, 5.001, 0.01), 2)`.
- `norm.pdf(x_grid, loc=mu, scale=SIGMA)` da $f(x \mid \mu)$ en toda la malla, y `np.maximum(x_grid, 0)` da $\hat\mu(x)$.
- `order = np.argsort(-R)` da los índices de mayor a menor $R$; `p[order]` y `x_grid[order]` reordenan los arreglos.
- `cum = np.cumsum(p[order])` acumula las probabilidades. Con `k = np.searchsorted(cum, CL)` toma los primeros `k + 1` puntos.
- Guarda el mínimo y el máximo de cada $A(\mu)$ en arreglos `x_lo` y `x_hi`, uno por cada $\mu$. La inversión es como en el ejercicio 2, y el cinturón se dibuja con `plt.fill_betweenx(mu_grid, x_lo, x_hi, alpha=0.3)`.
- Usa `plt.xlim(-4, 6)`. Para $\mu$ pequeño, $A(\mu)$ llega al borde de la malla, porque en realidad se extiende hasta $-\infty$.



In [ ]:
# Exercise 4 - Solution here